# ⚡ Strompreis-Analyse: Example Notebook

This minimal notebook shows how to use the core modules in `src/` to analyze your own electricity consumption data directly in Python.

In [1]:
import pandas as pd
from src.file_parser import ConsumptionDataParser
from src.data_loader import get_spot_data, merge_consumption_with_prices
from src.tariffs import TariffManager
from src.analysis import classify_usage, compute_cost_comparison_data

2026-09-04 16:34:42.730 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-09-04 16:34:42.731 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-09-04 16:34:42.731 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-09-04 16:34:42.733 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-09-04 16:34:42.733 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-09-04 16:34:42.734 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
/Users/matsschneider/miniconda3/envs/epex-analysis/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook 

## 1. Load and Parse Smart Meter Data

`ConsumptionDataParser` automatically detects and standardizes various European smart meter CSV formats into UTC timestamps and hourly/15-minute consumption.

In [2]:
parser = ConsumptionDataParser()
with open("resources/EXAMPLE-DATA-15M.csv", "rb") as f:
    df_consumption = parser.parse_file(f)

print(f"Loaded {len(df_consumption):,} records from {df_consumption['timestamp'].min().date()} to {df_consumption['timestamp'].max().date()}.")
df_consumption.head()

2026-09-04 16:34:42 [file_parser.py]: Loaded 7 user-defined provider configurations from cache/additional_provider_formats.json.
2026-09-04 16:34:43 [file_parser.py]: Loaded 35 provider configurations from cache.
2026-09-04 16:34:43 [file_parser.py]: Total of 42 provider formats loaded (7 user-defined, 35 main).
2026-09-04 16:34:43 [file_parser.py]: Successfully parsed with format: EXAMPLE DATA
2026-09-04 16:34:46 [file_parser.py]: Successfully localized naive timestamps using Europe/Vienna
2026-09-04 16:34:46 [file_parser.py]: Using aggregation level: 15min
2026-09-04 16:34:46 [file_parser.py]: Successfully resampled to 60284 rows
Loaded 60,284 records from 2022-11-03 to 2024-07-23.


,timestamp,consumption_kwh
0,2022-11-03 23:00:00+00:00,0.009
1,2022-11-03 23:15:00+00:00,0.027
2,2022-11-03 23:30:00+00:00,0.053
3,2022-11-03 23:45:00+00:00,0.007
4,2022-11-04 00:00:00+00:00,0.029


## 2. Fetch and Merge EPEX Spot Prices

Retrieve historical spot market data (cached locally in `cache/`) and align it with your consumption data.

In [3]:
# Use a 30-day window for demonstration
start_date = df_consumption["timestamp"].min().date()
end_date = (df_consumption["timestamp"].min() + pd.Timedelta(days=30)).date()

mask = (df_consumption["timestamp"].dt.date >= start_date) & (df_consumption["timestamp"].dt.date <= end_date)
df_sample = df_consumption.loc[mask]

# Fetch spot prices (Austria: 'at', Germany: 'de')
df_spot = get_spot_data(country="at", start=start_date, end=end_date)
df_merged = merge_consumption_with_prices(df_sample, df_spot)
df_merged.head()

2026-09-04 16:34:46.273 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-09-04 16:34:46.274 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-04 16:34:46.288 
  command:

    streamlit run /Users/matsschneider/miniconda3/envs/epex-analysis/lib/python3.12/site-packages/ipykernel_launcher.py [ARGUMENTS]
2026-09-04 16:34:46.288 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-04 16:34:46.288 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-04 16:34:46.289 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-04 16:34:46.358 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-04 1

2026-09-04 16:34:46 [data_loader.py]: Loading spot prices from cache.
2026-09-04 16:34:46 [data_loader.py]: Merging consumption data with spot prices.


,timestamp,consumption_kwh,spot_price_eur_kwh,date
0,2022-11-04 00:00:00+01:00,0.009,0.099408,2022-11-04
1,2022-11-04 00:15:00+01:00,0.027,0.099408,2022-11-04
2,2022-11-04 00:30:00+01:00,0.053,0.099408,2022-11-04
3,2022-11-04 00:45:00+01:00,0.007,0.099408,2022-11-04
4,2022-11-04 01:00:00+01:00,0.029,0.09438,2022-11-04


## 3. Compare Flexible vs. Static Tariffs

Calculate exact costs including provider markups, VAT, and prorated monthly fees.

In [4]:
manager = TariffManager("resources/tariffs_flexible.json", "resources/tariffs_static.json")
flex_tariffs = manager.get_flex_tariffs_with_custom()
static_tariffs = manager.get_static_tariffs_with_custom()

# Pick tariffs to compare (or configure your own Custom tariff)
flex_tariff = flex_tariffs["smartCONTROL (smartENERGY)"]
static_tariff = static_tariffs["V-Strom CLASSIC (VERBUND)"]

# Compute interval-by-interval costs
df_cost = manager.run_cost_analysis(df_merged.copy(), flex_tariff, static_tariff)

total_kwh = df_cost["consumption_kwh"].sum()
cost_flex = df_cost["total_cost_flexible"].sum()
cost_static = df_cost["total_cost_static"].sum()
savings = cost_static - cost_flex

print(f"Total Consumption: {total_kwh:.1f} kWh")
print(f"Static ({static_tariff.name}):   €{cost_static:.2f} (Ø €{cost_static / total_kwh:.3f}/kWh)")
print(f"Flexible ({flex_tariff.name}): €{cost_flex:.2f} (Ø €{cost_flex / total_kwh:.3f}/kWh)")
print(f"Potential Savings:                   €{savings:.2f} ({savings / cost_static * 100:+.1f}%)")

Total Consumption: 93.8 kWh
Static (V-Strom CLASSIC (VERBUND)):   €18.85 (Ø €0.201/kWh)
Flexible (smartCONTROL (smartENERGY)): €31.40 (Ø €0.335/kWh)
Potential Savings:                   €-12.55 (-66.6%)


## 4. Load Classification & Aggregated Summaries

Decompose consumption into base, regular, and peak components and aggregate costs across periods.

In [5]:
# Decompose usage into base, regular, and peak load
df_classified, base_thresh, peak_thresh = classify_usage(df_cost.copy(), "Europe/Vienna")
print(f"Base Load Total:    {df_classified['base_load_kwh'].sum():.1f} kWh")
print(f"Regular Load Total: {df_classified['regular_load_kwh'].sum():.1f} kWh")
print(f"Peak Load Total:    {df_classified['peak_load_kwh'].sum():.1f} kWh")

# Aggregate by week
weekly_summary = compute_cost_comparison_data(df_cost, resolution="Weekly")
weekly_summary

2026-09-04 16:34:46.415 No runtime found, using MemoryCacheStorageManager
2026-09-04 16:34:46.417 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-04 16:34:46.417 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-04 16:34:46.418 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-04 16:34:46.418 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-04 16:34:46.425 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-04 16:34:46.425 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-04 16:34:46.425 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


Base Load Total:    76.5 kWh
Regular Load Total: 10.0 kWh
Peak Load Total:    7.2 kWh
2026-09-04 16:34:46 [analysis.py]: Computing Cost Comparison Data


,timestamp,Total Consumption,Total Flexible Cost,Total Static Cost,Difference (€),Period,Avg. Static Price,Avg. Flex Price
0,2022-11-07 00:00:00+01:00,12.09,2.601174,2.452167,-0.149007,2022-W45,0.202826,0.215151
1,2022-11-14 00:00:00+01:00,20.436001,5.678254,4.183067,-1.495187,2022-W46,0.204691,0.277855
2,2022-11-21 00:00:00+01:00,21.737,7.28557,4.378217,-2.907353,2022-W47,0.201418,0.335169
3,2022-11-28 00:00:00+01:00,22.566,7.511589,4.502567,-3.009022,2022-W48,0.199529,0.332872
4,2022-12-05 00:00:00+01:00,16.966999,8.319124,3.33437,-4.984754,2022-W49,0.196521,0.490312
